In [ ]:
#1a below
import numpy as np
import itertools
from collections import defaultdict

N = 4

# possible increments
steps = [-1,1]

# store results
data = []

for alpha in [0,1]:
    for sigma in [0,1]:
        
        for dX in itertools.product(steps, repeat=N):
            for dY in itertools.product(steps, repeat=N):
                
                dX = np.array(dX)
                dY = np.array(dY)
                
                # random walks
                X = np.cumsum(dX)
                Y = np.cumsum(dY)
                
                # compute A
                A = (X % 5) - 2
                
                # increments of S
                dS = alpha*A + sigma*dY
                
                # compute S
                S = np.cumsum(dS)
                
                data.append({
                    "alpha":alpha,
                    "sigma":sigma,
                    "S":tuple(S)
                })

In [2]:
groups = defaultdict(list)

for row in data:
    groups[row["S"]].append(row)

In [3]:
F = {}
G = {}

for S, rows in groups.items():
    
    alpha_vals = [r["alpha"] for r in rows]
    sigma_vals = [r["sigma"] for r in rows]
    
    F[S] = round(np.mean(alpha_vals))
    G[S] = round(np.mean(sigma_vals))

In [4]:
correct_alpha = 0
correct_sigma = 0

for row in data:
    
    S = row["S"]
    
    if F[S] == row["alpha"]:
        correct_alpha += 1
        
    if G[S] == row["sigma"]:
        correct_sigma += 1

total = len(data)

print("P(correct alpha guess) =", correct_alpha/total)
print("P(correct sigma guess) =", correct_sigma/total)

P(correct alpha guess) = 0.9990234375
P(correct sigma guess) = 1.0


In [ ]:
import numpy as np
#part 1c below

# Parameters
N = 2026
steps = [-1, 1]
M = 100000

def simulate_one():
    # Simulate single step: X_N - X_{N-1} = ±1
    dX = np.random.choice(steps)
    dY = np.random.choice(steps)
    
    # Compute X_N incrementally (here only last step matters)
    # A_N = (X_N mod 5) - 2
    # Since we only need last step, A_N ∈ {-2,-1,0,1,2}
    # Approximate: let's just take dX = ±1
    # For simplicity, consider X_{N-1} = 0, so X_N = dX
    X_N = dX
    A_N = (X_N % 5) - 2  # this gives -1 or -1? check: (-1 %5) -2 = 4-2=2, (1%5)-2=-1 ✅
    
    # Compute ΔS_N
    delta_S_N = A_N + dY
    
    # Estimator: clip to [-2,2]
    R = np.clip(delta_S_N, -2, 2)
    
    # Squared error
    return (A_N - R)**2

errors = np.array([simulate_one() for _ in range(M)])
print("Estimated MSE for A_N estimator:", errors.mean())

Estimated MSE for A_N estimator: 0.74924
Estimated MSE for A_N estimator: 0.74924


In [14]:
#2a below

import numpy as np

M_lvl = 100000
A_vals = np.array([-2,-1,0,1,2])
Y_vals = np.array([-1,1])

W = []
for _ in range(M_lvl):
    A = np.random.choice(A_vals)
    dY = np.random.choice(Y_vals)
    W.append(A + dY)

# Compute probabilities
from collections import Counter
counts = Counter(W)
for k in sorted(counts):
    print(k, counts[k]/M_lvl)

-3 0.10146
-2 0.1002
-1 0.19988
0 0.20049
1 0.20003
2 0.09851
3 0.09943


In [15]:
#for 2b

import numpy as np
from itertools import product

# Parameters
N = 4
steps = [-1, 1]
A0 = -2  # given
alpha = 1
sigma = 1
M = 100000  # Monte Carlo iterations

def simulate_expected_profit():
    total_profit = 0.0

    # For each Monte Carlo iteration, simulate ΔX and ΔY sequences
    for _ in range(M):
        # Generate ΔX_1,...,ΔX_4
        dX_seq = np.random.choice(steps, size=N)
        # Generate ΔY_1,...,ΔY_4
        dY_seq = np.random.choice(steps, size=N)

        # Initialize X and Y
        X = np.zeros(N+1, dtype=int)
        Y = np.zeros(N+1, dtype=int)
        S = np.zeros(N+1, dtype=int)
        A = np.zeros(N+1, dtype=int)
        A[0] = A0

        # Compute X_n, Y_n, A_n
        for n in range(1, N+1):
            X[n] = X[n-1] + dX_seq[n-1]
            Y[n] = Y[n-1] + dY_seq[n-1]
            A[n] = (X[n] % 5) - 2
            # Compute ΔS_n = α*A_n + σ*(Y_n - Y_{n-1}) = A_n + ΔY_n
            delta_S = A[n] + (Y[n]-Y[n-1])
            # Optimal bet decision based on expectation: 
            # Since we don't know A_n, use expected value of ±1 X step for ΔX
            # Conditional expectation given past info (A_0 known) for ΔX_n
            # ΔX_n = ±1 equally likely → E[A_n | past] = ( ((X[n-1]+1)%5 -2) + ((X[n-1]-1)%5 -2) ) / 2
            expected_A = (( (X[n-1]+1)%5 -2) + ((X[n-1]-1)%5 -2 )) /2
            expected_delta_S = expected_A + 0  # ΔY_n has mean 0

            # Bet only if expected profit > 0
            if expected_delta_S > 0:
                total_profit += delta_S  # realized ΔS_n
            # else: do nothing, profit=0

    # Average over all iterations
    return total_profit / M

expected_profit = simulate_expected_profit()
print("Expected total profit over N=4:", expected_profit)

Expected total profit over N=4: 1.12946


In [16]:
#sim for 2c
import numpy as np

# Parameters
N = 2026
A0 = -2
steps = [-1,1]
M = 10000  # Monte Carlo iterations

def simulate_large_N_profit():
    total_profit = 0.0

    for _ in range(M):
        # Generate ΔX and ΔY sequences
        dX_seq = np.random.choice(steps, size=N)
        dY_seq = np.random.choice(steps, size=N)

        # Initialize
        X = np.zeros(N+1, dtype=int)
        Y = np.zeros(N+1, dtype=int)
        S = np.zeros(N+1, dtype=int)
        A = np.zeros(N+1, dtype=int)
        A[0] = A0

        profit = 0.0

        for n in range(1, N+1):
            X[n] = X[n-1] + dX_seq[n-1]
            Y[n] = Y[n-1] + dY_seq[n-1]
            A[n] = (X[n] % 5) - 2
            delta_S = A[n] + (Y[n]-Y[n-1])

            # Compute expected ΔS at decision time using only past info
            # For large N, conditional expectation ≈ 0
            expected_delta_S = 0  # justified by CLT + uniform mod 5

            if expected_delta_S > 0:
                profit += delta_S  # would bet
            # else: skip, profit = 0

        total_profit += profit

    return total_profit / M

expected_profit = simulate_large_N_profit()
print("Expected total profit over N=2026:", expected_profit)

Expected total profit over N=2026: 0.0
